In [ ]:
PAD=0  #padding id
BEG=1  #beginning id
END=2  #ending id
english=["the","cat","ate","rat","dog","mat","sat","log","on","big","ran","fast"]
coded=["dha","pil","thi","elu","kuk","cha","kur","kom","med","ped","par","veg"]   # corresponding code language words
eng_vocab={"<PAD>" :PAD,"<BEG>": BEG,"<END>":END}
cod_vocab={"<PAD>" :PAD,"<BEG>": BEG,"<END>":END}
for word in english:
  eng_vocab[word]=len(eng_vocab)
for word in coded:
  cod_vocab[word]=len(cod_vocab)
eng_id2word = {i: w for w, i in eng_vocab.items()}
cod_id2word = {i: w for w, i in cod_vocab.items()}
ENG_SIZE=len(eng_vocab)
COD_SIZE=len(cod_vocab)
print(ENG_SIZE)
print(COD_SIZE)  #english and coded language vocabulary size

15
15


In [ ]:
def enc_eng(sentence):
  return [eng_vocab.get(w) for w in sentence]   #convert senetnce to english encoding numbers
def enc_cod(sentence):
  return [cod_vocab.get(w) for w in sentence]   #convert senetnce to coded language encoding numbers
def dec_eng(ids):
    return [eng_id2word.get(i) for i in ids if i not in (PAD, BEG, END)]   # decoding into eng words
def dec_cod(ids) :
   return [cod_id2word.get(i) for i in ids if i not in (PAD, BEG, END)]   #decoding into code language words

In [ ]:
training_pairs = [
    #two words sentences
    (["cat",  "sat"],          ["pil", "kur"]),
    (["dog",  "ran"],          ["kuk", "par"]),
    (["rat",  "ate"],          ["elu", "thi"]),
    (["cat",  "ran"],          ["pil", "par"]),
    (["dog",  "sat"],          ["kuk", "kur"]),
    (["cat",  "ate"],          ["pil", "thi"]),

    # the + subject + verb
    (["the", "cat", "sat"],    ["dha", "pil", "kur"]),
    (["the", "dog", "ran"],    ["dha", "kuk", "par"]),
    (["the", "rat", "ate"],    ["dha", "elu", "thi"]),
    (["the", "cat", "ran"],    ["dha", "pil", "par"]),
    (["the", "dog", "sat"],    ["dha", "kuk", "kur"]),

    # subject + verb + object
    (["cat", "ate",  "rat"],   ["pil", "thi", "elu"]),
    (["dog", "ate",  "rat"],   ["kuk", "thi", "elu"]),
    (["rat", "ate",  "mat"],   ["elu", "thi", "cha"]),
    (["cat", "sat",  "on",  "mat"], ["pil", "kur", "med", "cha"]),
    (["dog", "ran",  "on",  "log"], ["kuk", "par", "med", "kom"]),
    (["rat", "ran",  "on",  "mat"], ["elu", "par", "med", "cha"]),

    # the + full sentence
    (["the", "cat", "sat", "on", "the", "mat"],
     ["dha", "pil", "kur", "med", "dha", "cha"]),

    (["the", "dog", "ran", "on", "the", "log"],
     ["dha", "kuk", "par", "med", "dha", "kom"]),

    (["the", "rat", "ate", "the", "mat"],
     ["dha", "elu", "thi", "dha", "cha"]),

    (["the", "cat", "ate", "the", "rat"],
     ["dha", "pil", "thi", "dha", "elu"]),

    # with big
    (["big",  "cat",  "sat"],       ["ped", "pil", "kur"]),
    (["big",  "dog",  "ran"],       ["ped", "kuk", "par"]),
    (["big",  "rat",  "ate"],       ["ped", "elu", "thi"]),
    (["big",  "cat",  "ate", "rat"],["ped", "pil", "thi", "elu"]),
    (["big",  "dog",  "sat", "on",  "log"],
     ["ped",  "kuk",  "kur", "med", "kom"]),

    # with fast
    (["fast", "cat",  "ran"],       ["veg", "pil", "par"]),
    (["fast", "dog",  "ran"],       ["veg", "kuk", "par"]),
    (["fast", "rat",  "ran"],       ["veg", "elu", "par"]),
    (["the",  "cat",  "ran", "fast"],["dha", "pil", "par", "veg"]),
    (["the",  "dog",  "ran", "fast"],["dha", "kuk", "par", "veg"]),

    # big + fast
    (["big",  "fast", "dog",  "ran"],["ped", "veg", "kuk", "par"]),
    (["big",  "fast", "cat",  "ran"],["ped", "veg", "pil", "par"]),

    # longer sentences
    (["the", "big",  "cat",  "sat", "on", "mat"],
     ["dha", "ped",  "pil",  "kur", "med", "cha"]),

    (["the", "fast", "dog",  "ran", "on", "log"],
     ["dha", "veg",  "kuk",  "par", "med", "kom"]),

    (["big", "rat",  "ran",  "on",  "log"],
     ["ped", "elu",  "par",  "med", "kom"]),

    (["the", "big",  "dog",  "ate", "the", "rat"],
     ["dha", "ped",  "kuk",  "thi", "dha", "elu"]),

    (["fast", "cat", "sat",  "on",  "the", "mat"],
     ["veg",  "pil", "kur",  "med", "dha", "cha"]),

    #  variety
    (["dog",  "sat",  "on",  "log"], ["kuk", "kur", "med", "kom"]),
    (["cat",  "ran",  "on",  "mat"], ["pil", "par", "med", "cha"]),
    (["rat",  "sat",  "on",  "log"], ["elu", "kur", "med", "kom"]),
    (["the",  "rat",  "ran", "fast"],["dha", "elu", "par", "veg"]),
    (["big",  "cat",  "ran", "fast"],["ped", "pil", "par", "veg"]),
]

print(f"Total number of  training pairs= {len(training_pairs)}")


Total number of  training pairs= 43


In [ ]:
import torch
SEQ_LEN=10
def prepare_training_data(pairs):  # preparing training data in format of transformer architecture for encoder and decoder seperate training
  enc_input=[]
  dec_input=[]
  dec_target=[]
  for i,j in pairs:
    enc_ids=enc_eng(i)
    dec_ids=enc_cod(j)  #raw ids after conversion word to numbers
    dec_inp=[BEG]+dec_ids
    dec_tgt=dec_ids+[END]   # including BEG and END for decoder training

    #padding to match SEQ_LEN
    enc_ids = enc_ids + [PAD] * (SEQ_LEN - len(enc_ids))
    dec_inp = dec_inp + [PAD] * (SEQ_LEN - len(dec_inp))
    dec_tgt = dec_tgt + [PAD] * (SEQ_LEN - len(dec_tgt))
    #appending each training set into main list
    enc_input.append(enc_ids)
    dec_input.append(dec_inp)
    dec_target.append(dec_tgt)
  return ( torch.tensor(enc_input), torch.tensor(dec_input),torch.tensor(dec_target))

enc_inputs, dec_inputs, dec_targets = prepare_training_data(training_pairs)  #loading actual training datas from training pairs
print(f"enc_inputs  shape: {enc_inputs.shape}")
print(f"dec_inputs  shape: {dec_inputs.shape}")
print(f"dec_targets shape: {dec_targets.shape}")


enc_inputs  shape: torch.Size([43, 10])
dec_inputs  shape: torch.Size([43, 10])
dec_targets shape: torch.Size([43, 10])


In [ ]:
def mask_1(enc):   #padding mask for encode self attention
  m = (enc != PAD)          #(B, seq_len)
  m = m.unsqueeze(1)    #(B, 1, seq_len)
  m = m.unsqueeze(2)    #(B, 1, 1,seq_len)
  return m

def mask_2(dec):
  m = (dec != PAD)          #(B,seq_len)
  m = m.unsqueeze(1)    #(B, 1, seq_len)
  m = m.unsqueeze(2)    #(B, 1, 1, seq_len)
  causal_mask = torch.tril(torch.ones(SEQ_LEN, SEQ_LEN))   #(seq_len, seq_len)
  causal_mask = causal_mask.unsqueeze(0)     # (1,seq_len, seq_len)
  causal_mask = causal_mask.unsqueeze(0)     # (1, 1,seq_len, seq_len)
  m=m & causal_mask.bool()   #(B,1,seq_len,seq_len)
  return m

In [ ]:
import math
def build_pe(max_seq_len, d_model):
    pe  = torch.zeros(max_seq_len, d_model)
    pos = torch.arange(0, max_seq_len).unsqueeze(1).float()
    div = torch.exp(
        torch.arange(0, d_model, 2).float()
        * (-math.log(10000.0) / d_model)
    )

    pe[:, 0::2] = torch.sin(pos * div)   # even dims
    pe[:, 1::2] = torch.cos(pos * div)   # odd dims
    return pe.unsqueeze(0)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class multi_head_attn(nn.Module):
  def __init__ (self,d_model,n_heads):
    super().__init__()
    self.d_model=d_model
    self.n_heads=n_heads
    self.head_dim=d_model//n_heads
    self.W_Q = nn.Linear(d_model, d_model)
    self.W_K = nn.Linear(d_model, d_model)
    self.W_V = nn.Linear(d_model, d_model)   # for query,key and values of self attn
    self.W_O = nn.Linear(d_model, d_model)   # for concatenating heads back to d_model
  def split_heads(self,x):
    B,seq_len,d_model=x.shape
    x = x.view(B, seq_len, self.n_heads, self.head_dim)  #dividing into heads in dimension
    x = x.transpose(1, 2)    #we want heads to do their attention seperately
    return x
  def combine_heads(self,x):
    B, n_heads, seq_len, head_dim = x.shape
    x = x.transpose(1, 2)
    x = x.contiguous().view(B, seq_len, self.d_model)  #just reverse of split heads
    return x
  def forward(self,Q,K,V,mask=None):
    Q = self.W_Q(Q)
    K = self.W_K(K)
    V = self.W_V(V)  # (B, seq_len, d_model)
    Q = self.split_heads(Q)    # (B, n_heads, seq_len, head_dim)
    K = self.split_heads(K)
    V = self.split_heads(V)
    scores = torch.matmul(Q, K.transpose(-2, -1))   #self attention
    scores = scores / math.sqrt(self.head_dim)
    if mask is not None:
      scores = scores.masked_fill(mask == 0, float('-inf'))
    attn_weights = F.softmax(scores, dim=-1)
    attn_out = torch.matmul(attn_weights, V)
    attn_out = self.combine_heads(attn_out)  #recombining heads after seperate attention
    attn_out = self.W_O(attn_out) # shape: (B, seq_len, d_model)
    return attn_out


In [ ]:
class ffn(nn.Module):
  def __init__(self,d_model,d_ff,dropout=0.1):
    super().__init__()
    self.ff1=nn.Linear(d_model,d_ff)
    self.ff2=nn.Linear(d_ff,d_model)
    self.dropout=nn.Dropout(dropout)
  def forward(self,x):
    x=self.ff1(x)   #expand
    x=F.relu(x)     # non-linearity
    x=self.dropout(x)   #regularisation
    x=self.ff2(x)   #compress back
    return x


In [ ]:
class encoder_block(nn.Module):
  def __init__(self,d_model,n_heads,d_ff,dropout=0.1):
    super().__init__()
    self.self_attn=multi_head_attn(d_model,n_heads)
    self.ff = ffn(d_model,d_ff,dropout)
    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 =nn.LayerNorm(d_model)
    self.dropout = nn.Dropout(dropout)
  def forward(self,x,m):
    attn_out =  self.self_attn(Q=x,K=x,V=x,mask=m)  #mask PaD tokens
    x=self.norm1(x+self.dropout(attn_out))    #residual + norm
    ff_out=self.ff(x)
    x=self.norm2(x+self.dropout(ff_out))
    return x

In [ ]:
class decoder_block(nn.Module):
  def __init__(self,d_model,n_heads,d_ff,dropout=0.1):
    super().__init__()
    self.self_attn=multi_head_attn(d_model,n_heads)
    self.cross_attn=multi_head_attn(d_model,n_heads)
    self.ff = ffn(d_model,d_ff,dropout)
    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 =nn.LayerNorm(d_model)
    self.norm3 =nn.LayerNorm(d_model)
    self.dropout = nn.Dropout(dropout)
  def forward(self,x,encoder_output,mask1,mask2):
    attn_out =  self.self_attn(Q=x,K=x,V=x,mask=mask2)  #causal mask tokens
    x=self.norm1(x+self.dropout(attn_out))    #residual + norm
    cross_out=self.cross_attn(Q=x,K=encoder_output,V=encoder_output,mask=mask1)  #cross attention
    x = self.norm2(x + self.dropout(cross_out))
    ff_out = self.ff(x)   #feed forward
    x = self.norm3(x+self.dropout(ff_out))
    return x

In [ ]:
class Transformer(nn.Module):
  def __init__(self,src_vocab_size,trg_vocab_size,d_model=64,n_heads=4,d_ff= 128,max_seq_len=10,dropout=0.1):
    super().__init__()
    self.d_model=d_model
    self.src_embedding = nn.Embedding(src_vocab_size, d_model)
    self.trg_embedding = nn.Embedding(trg_vocab_size, d_model)
    pe = build_pe(max_seq_len, d_model)
    self.register_buffer('pe', pe)
    self.encoder_block = encoder_block(d_model, n_heads, d_ff, dropout)
    self.decoder_block = decoder_block(d_model, n_heads, d_ff, dropout)
    self.output_head = nn.Linear(d_model, trg_vocab_size)
    self.dropout = nn.Dropout(dropout)
  def encode(self, src, src_mask):
      S = src.shape[1]
      x = self.src_embedding(src) * math.sqrt(self.d_model)
      x = self.dropout(x + self.pe[:, :S])
      x = self.encoder_block(x, src_mask)
      return x

  def decode(self, trg, encoder_out, src_mask, trg_mask):
      S = trg.shape[1]
      x = self.trg_embedding(trg) * math.sqrt(self.d_model)
      x = self.dropout(x + self.pe[:, :S])
      x = self.decoder_block(x, encoder_out, src_mask, trg_mask)
      return x

  def forward(self, src, tgt, src_mask, tgt_mask):
      encoder_out = self.encode(src, src_mask)
      decoder_out = self.decode(tgt, encoder_out, src_mask, tgt_mask)
      logits = self.output_head(decoder_out)
      return logits


In [ ]:
SRC_VOCAB_SIZE = len(eng_vocab)   # 15
TGT_VOCAB_SIZE = len(cod_vocab)   # 15
D_MODEL        = 64
N_HEADS        = 4
D_FF           = 128
MAX_SEQ_LEN    = 10
DROPOUT        = 0.1
# create model
model = Transformer(src_vocab_size = SRC_VOCAB_SIZE, trg_vocab_size = TGT_VOCAB_SIZE, d_model = D_MODEL,n_heads = N_HEADS,d_ff = D_FF,max_seq_len = MAX_SEQ_LEN,dropout = DROPOUT)


In [ ]:
import torch.optim as optim
EPOCHS = 1000
LR     = 1e-3
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = optim.AdamW(model.parameters(), lr=LR)
src_mask = mask_1(enc_inputs)
tgt_mask = mask_2(dec_inputs)

for epoch in range(EPOCHS):

    model.train()
    optimizer.zero_grad()    #clear old gradients
    logits = model(src = enc_inputs  ,tgt = dec_inputs,src_mask = src_mask,tgt_mask = tgt_mask)
    loss = criterion(logits.view(-1, TGT_VOCAB_SIZE),   dec_targets.view(-1))

    loss.backward()          #compute new gradients
    optimizer.step()         #update all parameters
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1:4d} and loss = {loss.item():.4f}")
print("\nTraining complete.")

Epoch  100 and loss = 0.0816
Epoch  200 and loss = 0.0189
Epoch  300 and loss = 0.0152
Epoch  400 and loss = 0.0078
Epoch  500 and loss = 0.0077
Epoch  600 and loss = 0.0071
Epoch  700 and loss = 0.0093
Epoch  800 and loss = 0.0050
Epoch  900 and loss = 0.0114
Epoch 1000 and loss = 0.0094

Training complete.


In [ ]:
def translate(sentence):
    model.eval()
    src_ids = enc_eng(sentence)
    src_ids = src_ids + [PAD] * (SEQ_LEN - len(src_ids))
    src = torch.tensor([src_ids])
    src_mask = mask_1(src)
    with torch.no_grad():
        encoder_out = model.encode(src, src_mask)

        dec_ids = [BEG]

        for _ in range(SEQ_LEN):
            dec_input = dec_ids + [PAD] * (SEQ_LEN - len(dec_ids))
            tgt = torch.tensor([dec_input])
            tgt_mask = mask_2(tgt)
            decoder_out = model.decode(
                tgt, encoder_out, src_mask, tgt_mask
            )


            logits = model.output_head(decoder_out)

            last_pos    = len(dec_ids) - 1
            next_logits = logits[0, last_pos, :]
            next_id     = next_logits.argmax().item()

            if next_id == END:
                break

            dec_ids.append(next_id)
    output_ids = dec_ids[1:]
    return dec_cod(output_ids)

In [ ]:

print("TRANSLATION TESTS")

# test on training sentences (should be perfect)
test_sentences = [
    ["cat",  "sat"],
    ["the",  "dog", "ran"],
    ["the",  "cat", "sat", "on", "the", "mat"],
    ["big",  "rat", "ate"],
    ["fast", "dog", "ran"],
    ["the",  "big", "cat", "sat", "on", "mat"],
]

for sentence in test_sentences:
    translation = translate(sentence)
    print(f"  {' '.join(sentence):35s} -> {' '.join(translation)}")

TRANSLATION TESTS
  cat sat                             -> pil kur
  the dog ran                         -> dha kuk par
  the cat sat on the mat              -> dha pil kur med dha cha
  big rat ate                         -> ped elu thi
  fast dog ran                        -> veg kuk par
  the big cat sat on mat              -> dha ped pil kur med cha
